# Igniters-Tunde-Wey Week 5 — Exercise Solutions

Solved exercises from **week5 Exercises**. Run from `week5` (parent) so paths to `knowledge-base/`, `vector_db`, and `evaluation` work.

---
## Exercise 1 (Day 1) — No key vs key in brute-force retrieval

**Task:** Ask the Day 1 app a question with no matching key, then with a key; compare responses.

**Solution:** When no word in the question is a key in `knowledge`, `get_relevant_context` returns `[]`. The system prompt then gets "There is no additional context..." and the LLM answers from its own knowledge (generic or wrong). When the query includes a key (e.g. "Carllm"), the matching document(s) are injected and the answer is grounded in the knowledge base.

In [ ]:
# Demonstrate the behavior without running the full Day 1 app:

knowledge_demo = {
    "carllm": "Carllm is an auto insurance product. Features: AI risk assessment, instant quoting.",
    "lancaster": "Avery Lancaster is CEO of Insurellm.",
}

def get_relevant_context_demo(message, knowledge):
    text = "".join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge[w] for w in words if w in knowledge]

# No key match: "company" and "main" and "product" are not keys
ctx_no_key = get_relevant_context_demo("What is the company's main product?", knowledge_demo)
print("Query: What is the company's main product?")
print(f"Context retrieved: {len(ctx_no_key)} doc(s) -> {ctx_no_key}")

# With key: "carllm" is a key
ctx_with_key = get_relevant_context_demo("What is Carllm?", knowledge_demo)
print("\nQuery: What is Carllm?")
print(f"Context retrieved: {len(ctx_with_key)} doc(s) -> {ctx_with_key[0][:80]}...")

print("\n=> Day 2 improves this with embeddings so semantic similarity can surface relevant docs.")

---
## Exercise 2 (Day 2) — Change chunk_size and compare

**Task:** Change the text splitter's `chunk_size`, re-run chunking and Chroma, then in Day 3 see if answers or retrieved chunks change.

**Solution:** In `../day2.ipynb`, the chunking cell uses `RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)`. Try e.g. `chunk_size=400` or `chunk_size=1500`, then re-run the splitter and the Chroma `from_documents` cell (and delete the old DB first if needed). In Day 3, the same question may retrieve different chunks: smaller chunks give more precise hits but more fragments; larger chunks give more context per chunk but can be less targeted.

In [ ]:
# Snippet to use in day2.ipynb for an alternative chunk_size (run from week5):

# Option A: smaller chunks (more precise retrieval, more fragments)
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=100)

# Option B: larger chunks (more context per chunk, less targeted)
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)

# Then: chunks = text_splitter.split_documents(documents)
# Re-build Chroma (delete vector_db first or use a new db_name to compare).

print("After re-building the vector DB in day2, open day3 and ask the same question.")
print("Compare: number of chunks retrieved, and whether the answer is more/less accurate.")

---
## Exercise 3 (Day 3) — Paraphrased questions and semantic retrieval

**Task:** Ask questions that use different wording than the documents; see if the retriever still returns the right chunks. Compare with Day 1.

**Solution:** With vector retrieval, paraphrased questions often still retrieve the right chunks because embeddings capture semantic similarity. Examples: "Who runs the company?" / "Who is the CEO?" both map to CEO-related content; "What do we sell for cars?" can map to Carllm. In Day 1, only exact key words (e.g. "lancaster", "carllm") would match; in Day 3, meaning matters.

In [ ]:
# Example paraphrased questions to try in the Day 3 Gradio app:

paraphrased = [
    "Who runs the company?",           # -> CEO / Avery Lancaster
    "What is our auto insurance product?",  # -> Carllm
    "Who got the top award in 2023?", # -> IIOTY / Maxine Thompson
]

print("Try these in Day 3 RAG UI and check that retrieved chunks and answers are correct.")
for q in paraphrased:
    print(f"  - {q}")

print("\nIn Day 1, none of these would match a key (no 'lancaster', 'carllm', etc.).")
print("In Day 3, embeddings allow semantic match (e.g. 'runs the company' ~ 'CEO').")

---
## Exercise 4 (Day 4) — Analyze a failing/borderline case and suggest a change

**Task:** Pick one failing or borderline test, run RAG manually, inspect retrieved chunks, and suggest one change.

**Solution:** Load tests, run retrieval and answer evaluation for a few tests, find one with low MRR or low accuracy/completeness. Inspect what was retrieved; if the right doc wasn't in the top-k, try increasing `k` or improving chunking; if the right doc was there but the answer was wrong, improve the prompt or model.

In [ ]:
# Run from week5 so 'evaluation' and 'implementation' are on the path.
import sys
from pathlib import Path

week5 = Path("..").resolve() if Path("..").resolve().name == "week5" else Path(".").resolve()
if str(week5) not in sys.path:
    sys.path.insert(0, str(week5))

from evaluation.test import load_tests
from evaluation.eval import evaluate_retrieval, evaluate_answer

tests = load_tests()

# Find a borderline or failing case (e.g. low retrieval or low answer score)
results = []
for i, test in enumerate(tests[:20]):  # first 20
    ret = evaluate_retrieval(test)
    ans_eval, answer, chunks = evaluate_answer(test)
    results.append((i, test, ret, ans_eval, answer, chunks))

# Show one with low keyword coverage or low accuracy
for i, test, ret, ans_eval, answer, chunks in results:
    if ret.keyword_coverage < 80 or ans_eval.accuracy < 4.0:
        print(f"Test #{i}: {test.question}")
        print(f"  Category: {test.category}")
        print(f"  Retrieval: MRR={ret.mrr:.3f}, keyword_coverage={ret.keyword_coverage:.0f}%")
        print(f"  Answer scores: accuracy={ans_eval.accuracy}, completeness={ans_eval.completeness}")
        print(f"  Reference: {test.reference_answer[:100]}...")
        print(f"  Retrieved chunks (first 2):")
        for j, doc in enumerate(chunks[:2]):
            print(f"    [{j}] {doc.page_content[:150]}...")
        print(f"  Suggested change: If keywords missing -> increase top_k or chunk_size. If answer wrong but context OK -> improve system prompt or add few-shot.")
        break
else:
    print("No clearly failing case in first 20; pick any test and inspect retrieval + answer.")

---
## Exercise 5 (Day 5) — Compare Day 3 vs Day 5 pipeline with the eval set

**Task:** Run the Day 4 evaluator (or a subset) on the Day 5 pipeline and compare with Day 3.

**Solution:** The standard `evaluation.eval` uses `implementation.answer`, which is wired to `vector_db` (Day 3). To compare Day 3 vs Day 5 we run retrieval (and optionally answer) for a subset of tests using each vector store. Day 3 DB uses HuggingFace `all-MiniLM-L6-v2`; Day 5 `preprocessed_db` uses OpenAI `text-embedding-3-large`. We compute retrieval metrics (e.g. keyword coverage) for each and compare.

In [ ]:
# Run from week5 directory. Compares retrieval quality (keyword coverage) for Day 3 vs Day 5 DBs.
import sys
from pathlib import Path

week5 = Path("..").resolve() if (Path("..") / "evaluation").exists() else Path(".").resolve()
if not (week5 / "evaluation").exists():
    week5 = Path(".").resolve()
sys.path.insert(0, str(week5))

from evaluation.test import load_tests
from evaluation.eval import calculate_mrr
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings

tests = load_tests()[:30]  # subset for speed

# Day 3: vector_db with HuggingFace (same as day2)
db_day3 = week5 / "vector_db"
emb_day3 = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vs_day3 = Chroma(persist_directory=str(db_day3), embedding_function=emb_day3)
retriever_day3 = vs_day3.as_retriever(search_kwargs={"k": 10})

def retrieval_metrics(retriever, test):
    docs = retriever.invoke(test.question)
    from langchain_core.documents import Document
    if not docs:
        return 0.0, 0.0
    mrr_scores = [calculate_mrr(kw, docs) for kw in test.keywords]
    avg_mrr = sum(mrr_scores) / len(mrr_scores) if mrr_scores else 0.0
    coverage = 100 * sum(1 for s in mrr_scores if s > 0) / len(test.keywords) if test.keywords else 0
    return avg_mrr, coverage

mrr_3, cov_3 = [], []
for t in tests:
    m, c = retrieval_metrics(retriever_day3, t)
    mrr_3.append(m)
    cov_3.append(c)

print("Day 3 (vector_db, HuggingFace):")
print(f"  Avg MRR: {sum(mrr_3)/len(mrr_3):.4f}")
print(f"  Avg keyword coverage: {sum(cov_3)/len(cov_3):.1f}%")

# Day 5: preprocessed_db (only if it exists; uses OpenAI embeddings)
db_day5 = week5 / "preprocessed_db"
if db_day5.exists():
    emb_day5 = OpenAIEmbeddings(model="text-embedding-3-large")
    vs_day5 = Chroma(persist_directory=str(db_day5), embedding_function=emb_day5)
    retriever_day5 = vs_day5.as_retriever(search_kwargs={"k": 10})
    mrr_5, cov_5 = [], []
    for t in tests:
        m, c = retrieval_metrics(retriever_day5, t)
        mrr_5.append(m)
        cov_5.append(c)
    print("Day 5 (preprocessed_db, OpenAI):")
    print(f"  Avg MRR: {sum(mrr_5)/len(mrr_5):.4f}")
    print(f"  Avg keyword coverage: {sum(cov_5)/len(cov_5):.1f}%")
    print("\nConclusion: Compare metrics; LLM chunking (Day 5) often improves retrieval for semantic queries.")
else:
    print("Day 5 preprocessed_db not found. Run day5.ipynb to build it, then re-run this cell.")

---
## Summary

| Exercise | Main takeaway |
|----------|----------------|
| 1 | Brute-force retrieval only matches query words that are keys; no key → no context → generic/wrong answer. |
| 2 | `chunk_size` trades off precision (smaller) vs context per chunk (larger); re-build Chroma after changing. |
| 3 | Vector retrieval supports paraphrased questions; Day 1 keyword match does not. |
| 4 | For failures: inspect retrieval first (right doc in top-k?); then prompt/LLM. |
| 5 | Compare Day 3 vs Day 5 retrieval (and optionally answer) on the same test set to see if LLM chunking helps. |